# Stage 1: Pretrain (TinyLlama 154M, wikitext-103 + code, 7:3)

Воспроизведение TinyLlama (arXiv:2401.02385): Llama-2 архитектура из блоков
`spartan_torch` (RoPE + RMSNorm + SwiGLU + GQA 8:1), полный дата-рецепт
NL:code ≈ 7:3. Начало цепочки стадий: pretrain → continue_pretrain → cooldown.

Запуск: ноутбук открывается из каталога `1.pretrain` (cwd = `1.pretrain`).
Общие кэши (токенизатор, блоки) — в `../data/`, переиспользуются стадиями 2 и 3.

## Отклонения от статьи (для будущего воспроизведения на GPU)

| Параметр | У нас | Статья | Примечание |
|---|---|---|---|
| vocab_size | 8192 | 32000 | Llama-токенизатор, 4× больше |
| d_model | 1024 | 2048 | ×1/2 |
| n_layer | 18 | 22 | ×0.82 |
| n_head | 16 | 32 | ×1/2 |
| num_kv_heads | 2 | 4 | GQA 8:1 сохранён |
| ff_hidden_size | 2816 | 5632 | пропорция 2.75× сохранена |
| block_size | 512 | 2048 | ×1/4 |
| NL данные | wikitext-103 | SlimPajama | другой корпус |
| Code данные | codeparrot-clean | StarCoder | StarCoder gated |
| Токенов | ~500M | ~3T | ×6 |
| Warmup | 500 | 2000 | пропорционально |
| min_lr | 1e-5 | ~0 | не даём LR провалиться |
| Dropout | 0 | 0 | = |
| Tied embeddings | да | да | = |
| Residual init | 1/√(2·n_layer) | 1/√(2·n_layer) | = |

## 0. Настройки

In [8]:
from pathlib import Path
import sys

import torch

ROOT = Path.cwd()
EXPERIMENT_ROOT = ROOT.parent
sys.path.insert(0, str(EXPERIMENT_ROOT))  # tinyllama/

DATA_DIR = EXPERIMENT_ROOT / "data"
CKPT_DIR = ROOT / "checkpoints"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"cwd: {ROOT} | device: {DEVICE}")

SEED = 0
torch.manual_seed(SEED)

# --- модель (154M, пропорции TinyLlama: GQA 8:1, ff = 2.75x, без dropout) ---
VOCAB_SIZE = 8192
D_MODEL = 1024
N_LAYER = 18
N_HEAD = 16
NUM_KV_HEADS = 2
FF_HIDDEN_SIZE = 2816
BLOCK_SIZE = 512
DROPOUT_P = 0.0

# --- данные: 70% wikitext-103 + 30% codeparrot-clean ---
MIX = "pretrain"
TOTAL_BLOCKS = 975000

# --- тренер ---
EPOCHS = 3
# 4 GB VRAM: batch 32 не влезает (модель fp32 + AdamW m+v + активации) —
# драйвер молча льёт в общую RAM и шаг падает с ~5 c до 20+. Батч 8 x accum 4
# даёт тот же эффективный батч без spill.
BATCH_SIZE = 8
GRAD_ACCUM = 4
SANITY_MAX_STEPS = 200  # sanity-прогон; None = полный запуск по EPOCHS
LR = 3e-4
WEIGHT_DECAY = 0.1
WARMUP_STEPS = 10
MIN_LR = 1e-5
EVAL_STEPS = 50
SAVE_STEPS = 50

# --- MLflow ---
MLFLOW_TRACKING_URI = "http://host.docker.internal:5000"
MLFLOW_EXPERIMENT_NAME = "tinyllama-pretrain"

torch.set_float32_matmul_precision("medium")

cwd: /workspaces/spartan-torch/experiments/llm/tinyllama/1.pretrain | device: cuda


## 1. Данные: токенизатор на смешанном корпусе + блоки-кэши + микс 7:3

In [9]:
from data import (
    build_eval_dataset,
    build_tokenizer,
    iter_texts,
    load_tokenizer,
    make_mixed_dataset,
)

TOKENIZER_PATH = DATA_DIR / "tokenizer"
if TOKENIZER_PATH.exists():
    print("tokenizer: cached")
    tokenizer = load_tokenizer(TOKENIZER_PATH)
else:
    print("tokenizer: training byte-level BPE on mixed corpus")
    seed_texts = (
        iter_texts("nl", 3000)
        + iter_texts("code", 1500)
        + iter_texts("math", 500)
    )
    tokenizer = build_tokenizer(seed_texts, TOKENIZER_PATH, vocab_size=VOCAB_SIZE)

train_ds = make_mixed_dataset(
    MIX,
    DATA_DIR / "blocks",
    tokenizer,
    BLOCK_SIZE,
    total_blocks=TOTAL_BLOCKS,
    seed=SEED,
)
val_ds = build_eval_dataset(tokenizer, BLOCK_SIZE, cache_dir=DATA_DIR / "blocks")
print(f"vocab={tokenizer.vocab_size} | train blocks={len(train_ds):,} | eval blocks={len(val_ds):,}")

tokenizer: cached


[data] pretrain: nl 682,500 blocks (weight 0.70)
[data] pretrain: code 292,500 blocks (weight 0.30)
[data] pretrain: 975,000 blocks total
vocab=8192 | train blocks=975,000 | eval blocks=685


## 2. Модель и конфиг

In [10]:
from model import CausalLM, TinyLlamaConfig

cfg = TinyLlamaConfig(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    num_kv_heads=NUM_KV_HEADS,
    ff_hidden_size=FF_HIDDEN_SIZE,
    block_size=BLOCK_SIZE,
    dropout_p=DROPOUT_P,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

model = CausalLM(cfg)
x = torch.randint(0, VOCAB_SIZE, (2, BLOCK_SIZE))
with torch.no_grad():
    loss = model(input_ids=x, labels=x).loss
assert loss is not None
params_mb = model.num_params() * 2 / 1e6  # bf16
print(f"params: {model.num_params():,} | fwd ok | random-init loss={loss.item():.3f}")
print(f"model bf16 ~ {params_mb:.0f} MB + AdamW m+v fp32 ~ {model.num_params() * 8 / 1e6:.0f} MB")

params: 206,625,792 | fwd ok | random-init loss=9.210
model bf16 ~ 413 MB + AdamW m+v fp32 ~ 1653 MB


## 3. HF Trainer

In [11]:
from transformers import DataCollatorForLanguageModeling, EarlyStoppingCallback, Trainer, TrainingArguments

import mlflow
from train import SampleTextCallback, report_to_value

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

args = TrainingArguments(
    output_dir=str(CKPT_DIR),
    num_train_epochs=EPOCHS,
    max_steps=SANITY_MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine_with_min_lr",
    lr_scheduler_kwargs={"min_lr": MIN_LR},
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=20,
    bf16=(DEVICE == "cuda"),
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    torch_compile=(DEVICE == "cuda"),
    seed=SEED,
    report_to=report_to_value(MLFLOW_TRACKING_URI),
    run_name=MLFLOW_EXPERIMENT_NAME,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)

sample_cb = SampleTextCallback(
    model,
    tokenizer,
    prompts=["The history of Rome", "Machine learning is", "def fibonacci(n):"],
    max_new_tokens=64,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    callbacks=[sample_cb, EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

🏃 View run tinyllama-pretrain at: http://host.docker.internal:5000/#/experiments/4/runs/2780c194ac644cc2a8f6a294a9ccbeaf
🧪 View experiment at: http://host.docker.internal:5000/#/experiments/4


Step,Training Loss,Validation Loss
50,6.909815,6.828782
100,6.726895,6.556169
150,6.370454,6.131660
200,6.065784,5.993874


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


### prompt
The history of Rome
### completion
The history of Rome = = = = = Al 19ack = 
 = = 
 
 = = = 
 T D H = She Ps 
 = 
 = = = 
 Reb M Rribad = = = = = D Krian Aor = 
 = = = = = = = = = Pro. "

### prompt
Machine learning is
### completion
Machine learning is with the new , Hb ) ( be a second a gel4 in 1 @.@ a Gorzes fely Mis . 
 = = = To the reack = = = Hawers in 18. Luafuers by the a in S. H. sre = =

### prompt
def fibonacci(n):
### completion
def fibonacci(n):
       .1. ' ".w is.c,
    2.  :
            w is._ [ d.x, '
     , #):
        self.d):
        d the list the list %._

### of a m.


O
	D.#




[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


🏃 View run tinyllama-pretrain at: http://host.docker.internal:5000/#/experiments/4/runs/130f234c78044020bb8929e9b847843e
🧪 View experiment at: http://host.docker.internal:5000/#/experiments/4


TrainOutput(global_step=200, training_loss=6.646844635009765, metrics={'train_runtime': 4079.0539, 'train_samples_per_second': 1.569, 'train_steps_per_second': 0.049, 'total_flos': 3897501627187200.0, 'train_loss': 6.646844635009765, 'epoch': 0.006564102564102564})

## 4. Проверка

In [12]:
from train import evaluate_ppl, generate

model = trainer.model  # load_best_model_at_end подставил лучший
trainer.save_model(str(CKPT_DIR / "best"))
print("best ckpt:", CKPT_DIR / "best")

ppl = evaluate_ppl(model, val_ds, tokenizer, device=DEVICE)
print(f"val perplexity: {ppl:.2f}")

print("\nsample (text):")
print(generate(model, tokenizer, "The history of Rome", max_new_tokens=48))
print("\nsample (code):")
print(generate(model, tokenizer, "def fibonacci(n):", max_new_tokens=48))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

best ckpt: /workspaces/spartan-torch/experiments/llm/tinyllama/1.pretrain/checkpoints/best
val perplexity: 400.96

sample (text):
The history of Rome = P nleianries Friy , the Bitfs and 19rely sener catamizens and his uning to Graes with this Ning the game of the Test

sample (code):
def fibonacci(n):2, 2 to the  used, is 3 is re. 1 in be a 1, in the first cg, Car of this 2 of 5 for 1 to the to one to her " for the revel ) of the
